**Figure 2 A : Reaction Count Boxplot**

In [ ]:
import os
import scipy.io as sio
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


folder_FASTCORE = r'D:\context specific models\FASTCOREmodels' # model outpu data FASTCOREmodels path
tissue_names = {
    'BL': 'BronchusLung',
    'B': 'Breast',
    'C': 'Colon',
    'T': 'Thyroid',
    'S': 'Stomach',
    'P': 'Prostate',
    'K': 'Kidney',
    'LI': 'Liver'
}

# Function to load model and get reaction count
def get_reaction_count(folder, tissue_model_folder):
    folder_path = os.path.join(folder, tissue_model_folder)
    rxn_counts = []
    if os.path.exists(folder_path):
        for file_name in os.listdir(folder_path):
            if file_name.endswith('.mat'):
                file_path = os.path.join(folder_path, file_name)
                data = sio.loadmat(file_path)
                if 'model' in data:
                    rxn_count = len(data['model']['rxns'][0][0])
                    rxn_counts.append(rxn_count)
                else:
                    print(f"No 'model' structure found in {file_path}")
    else:
        print(f"Folder not found: {folder_path}")
    return rxn_counts


boxplot_data = []

# Iterate over subfolders in FASTCOREmodels
for tissue_model_folder in os.listdir(folder_FASTCORE):
    if os.path.isdir(os.path.join(folder_FASTCORE, tissue_model_folder)):
        fastcore_rxn_counts = get_reaction_count(folder_FASTCORE, tissue_model_folder)
        condition = 'Normal' if tissue_model_folder.endswith('N') else 'Tumor'
        tissue_abbr = tissue_model_folder[:-1]
        tissue = tissue_names.get(tissue_abbr, tissue_abbr)
        for rxn_count in fastcore_rxn_counts:
            boxplot_data.append({
                'Tissue': tissue,
                'Condition': condition,
                'Reaction Count': rxn_count
            })


df = pd.DataFrame(boxplot_data)

# Perform statistical test (t-test) for each tissue
p_values = []
for tissue in df['Tissue'].unique():
    normal = df[(df['Tissue'] == tissue) & (df['Condition'] == 'Normal')]['Reaction Count']
    tumor = df[(df['Tissue'] == tissue) & (df['Condition'] == 'Tumor')]['Reaction Count']
    t_stat, p_value = stats.ttest_ind(normal, tumor)
    p_values.append((tissue, p_value))

# Create box plot
plt.figure(figsize=(15, 10))
ax = sns.boxplot(x='Tissue', y='Reaction Count', hue='Condition', data=df,
                 palette={'Normal': 'lightblue', 'Tumor': 'lightcoral'})


y_max = df['Reaction Count'].max()
for i, (tissue, p_value) in enumerate(p_values):
    x1, x2 = i - 0.2, i + 0.2
    y = y_max + 50
    plt.plot([x1, x1, x2, x2], [y, y + 20, y + 20, y], lw=1.5, c='black')
    significance_marker = '*' if p_value < 0.05 else 'NS'
    plt.text((x1 + x2) * 0.5, y + 25, significance_marker, ha='center', va='bottom')

plt.title('Comparison of Reaction Counts Across Tissues and Conditions',fontsize=17)
plt.xlabel('Tissue',fontsize=16)
plt.ylabel('Number of Reactions',fontsize=15)
plt.xticks(rotation=45, ha="right",fontsize=14)
plt.yticks(fontsize=14)
plt.legend(title='Condition')


output_file = r'D:\context specific models\tissue_reaction_count_comparison_boxplot_with_significance.png'
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"Plot saved as: {output_file}")
plt.show()

#printing summary of stats
print("\nSummary Statistics:")
print(df.groupby(['Tissue', 'Condition'])['Reaction Count'].describe())
print("\nSignificance Test Results:")
for tissue, p_value in p_values:
    significance = "Significant" if p_value < 0.05 else "Non-significant"
    print(f"{tissue}: p-value = {p_value:.4f} ({significance})")

**Filtering the Reactions to 80% cutoff** (**inputdata - from model stats** - core reactions coloumns were takin for cancer normal both saved in a dataframe description is below and cuttoff of 80% applied results were saved back as filtered coloums in same sheet now named **" 80%_ cutoff reactions"** )

In [ ]:
import pandas as pd

def filter_reactions_by_cutoff(df, reaction_column, model_count_column, total_models, cutoff_percentage=80):
    """
    Filters reactions that are present in at least the specified cutoff percentage of models.

    Parameters:
    - df (DataFrame): The DataFrame containing reactions and model counts.
    - reaction_column (str): The column name for reactions.
    - model_count_column (str): The column name for model counts.
    - total_models (int): The total number of models for cutoff calculation.
    - cutoff_percentage (float): The cutoff percentage to filter reactions (default is 80).

    Returns:
    - Series: A Series containing reactions that meet the cutoff condition.
    """
    cutoff_count = total_models * (cutoff_percentage / 100)
    return df[reaction_column].where(df[model_count_column] >= cutoff_count)

# User inputs
file_path = input("Enter the Excel file path: ")
sheet_name = input("Enter the sheet name: ")
total_models_normal = int(input("Enter the total number of models for Core Normal: "))
total_models_cancer = int(input("Enter the total number of models for Core Cancer: "))

core_normal_reaction_column = 'Core normal'
core_normal_model_count_column = 'Core normal count'
core_cancer_reaction_column = 'Core cancer'
core_cancer_model_count_column = 'Core cancer count'
df = pd.read_excel(file_path, sheet_name=sheet_name)

# Apply filtering based on the 80% cutoff for each column
df['Filtered_Core_Normal'] = filter_reactions_by_cutoff(
    df, core_normal_reaction_column, core_normal_model_count_column, total_models_normal
)

df['Filtered_Core_Cancer'] = filter_reactions_by_cutoff(
    df, core_cancer_reaction_column, core_cancer_model_count_column, total_models_cancer
)

filtered_normal_reactions = df['Filtered_Core_Normal'].dropna().unique()
filtered_cancer_reactions = df['Filtered_Core_Cancer'].dropna().unique()

# Identify unique reactions to each filtered list
unique_reactions_core_normal = list(set(filtered_normal_reactions) - set(filtered_cancer_reactions))
unique_reactions_core_cancer = list(set(filtered_cancer_reactions) - set(filtered_normal_reactions))

max_length = max(len(unique_reactions_core_normal), len(unique_reactions_core_cancer))
unique_df = pd.DataFrame({
    'Unique_Reactions_Core_Normal': unique_reactions_core_normal + [None] * (max_length - len(unique_reactions_core_normal)),
    'Unique_Reactions_Core_Cancer': unique_reactions_core_cancer + [None] * (max_length - len(unique_reactions_core_cancer))
})


df = pd.concat([df, unique_df], axis=1)
with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df.to_excel(writer, sheet_name=sheet_name, index=False)

print("Filtered and unique reactions for Core Normal and Core Cancer have been saved to the Excel file.")


Enter the Excel file path: D:\context specific models\Core_rxns.xlsx
Enter the sheet name: BronchusLung
Enter the total number of models for Core Normal: 26
Enter the total number of models for Core Cancer: 23
Filtered and unique reactions for Core Normal and Core Cancer have been saved to the Excel file.


**CORE , ACCESSORY AND UNIQUE ACROSS ALL MODELS NORMAL AND CANCER** ( data used from the filtered coloumns of 80% cuttoff sheet (not used )

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


file_path = r'file path'# 80% cuttoff file titled in data used
sheets = ['BronchusLung', 'Breast', 'Colon', 'Thyroid', 'Stomach', 'Prostate', 'Kidney', 'Liver']
sheets_data = {sheet: pd.read_excel(file_path, sheet_name=sheet) for sheet in sheets}
reactions_by_sheet = {
    sheet: set(sheets_data[sheet]["Filtered_Core_Normal"].dropna()).union(
        sheets_data[sheet]["Filtered_Core_Cancer"].dropna()
    )
    for sheet in sheets
}

# Determine core, accessory, and unique reactions
all_reactions = set.union(*reactions_by_sheet.values())
core_reactions = set.intersection(*reactions_by_sheet.values())

reaction_sets = {}
for sheet, reactions in reactions_by_sheet.items():
    # Unique reactions: Only present in this tissue
    unique_reactions = reactions - set.union(*[reactions_by_sheet[s] for s in sheets if s != sheet])

    # Accessory reactions: Shared with at least one other tissue but not in all tissues
    accessory_reactions = reactions - core_reactions - unique_reactions


    reaction_sets[sheet] = {
        "Core": core_reactions,
        "Accessory": accessory_reactions,
        "Unique": unique_reactions
    }

# Save the reaction sets into separate DataFrames using openpyxl
from openpyxl import load_workbook

with pd.ExcelWriter(file_path, mode="a", engine="openpyxl", if_sheet_exists="overlay") as writer:
    for sheet, sets in reaction_sets.items():
        core_df = pd.DataFrame({"Core Reactions": list(sets["Core"])})
        accessory_df = pd.DataFrame({"Accessory Reactions": list(sets["Accessory"])})
        unique_df = pd.DataFrame({"Unique Reactions": list(sets["Unique"])})
        core_df.to_excel(writer, sheet_name=f"{sheet}_Core", index=False)
        accessory_df.to_excel(writer, sheet_name=f"{sheet}_Accessory", index=False)
        unique_df.to_excel(writer, sheet_name=f"{sheet}_Unique", index=False)

# Prepare data for the stacked bar plot
plot_data = {
    "Tissue": [],
    "Core": [],
    "Accessory": [],
    "Unique": []
}
for sheet, sets in reaction_sets.items():
    plot_data["Tissue"].append(sheet)
    plot_data["Core"].append(len(sets["Core"]))
    plot_data["Accessory"].append(len(sets["Accessory"]))
    plot_data["Unique"].append(len(sets["Unique"]))

# Convert to dataframe for plotting
plot_df = pd.DataFrame(plot_data)

plt.figure(figsize=(12, 8), facecolor='white')
plt.gcf().patch.set_facecolor('white')
bottom = None
for category, color in zip(["Core", "Accessory", "Unique"], ["#1f77b4", "#ff7f0e", "#2ca02c"]):
    if bottom is None:
        plt.bar(plot_df["Tissue"], plot_df[category], label=category, color=color)
        bottom = plot_df[category]
    else:
        plt.bar(plot_df["Tissue"], plot_df[category], bottom=bottom, label=category, color=color)
        bottom += plot_df[category]


plt.xlabel("Tissue", fontsize=16)
plt.ylabel("Number of Reactions", fontsize=15)
plt.title("Core, Accessory, and Unique Reactions Across Tissues", r )
plt.legend(loc="upper left", fontsize=12)
plt.xticks(rotation=45, fontsize=14)
plt.yticks(fontsize=14)
plt.tight_layout()

ax = plt.gca()
ax.spines['bottom'].set_color('black')
ax.spines['left'].set_color('black')
ax.spines['bottom'].set_linewidth(1.5)
ax.spines['left'].set_linewidth(1.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='x', colors='black')
ax.tick_params(axis='y', colors='black')
plt.grid(False)
plt.gca().set_facecolor('white')


plt.savefig('Stack core cancer unique.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

**CORE , ACCESSORY AND UNIQUE ACROSS ALL MODELS Only CANCER (PAnel 2 Fig B)**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


file_path = r'D:\context specific models\Core_rxns.xlsx'   # Replace with your file path which is 80% cuttoff file
sheets = ['BronchusLung', 'Breast', 'Colon', 'Thyroid', 'Stomach', 'Prostate', 'Kidney', 'Liver']


sheets_data = {sheet: pd.read_excel(file_path, sheet_name=sheet) for sheet in sheets}
reactions_by_sheet = {
    sheet: set(sheets_data[sheet]["Filtered_Core_Cancer"].dropna())
    for sheet in sheets
}

# Determine core, accessory, and unique reactions
all_reactions = set.union(*reactions_by_sheet.values())
core_reactions = set.intersection(*reactions_by_sheet.values())

reaction_sets = {}
for sheet, reactions in reactions_by_sheet.items():
    # Unique reactions: Only present in this tissue
    unique_reactions = reactions - set.union(*[reactions_by_sheet[s] for s in sheets if s != sheet])

    # Accessory reactions: Shared with at least one other tissue but not in all tissues
    accessory_reactions = reactions - core_reactions - unique_reactions
    reaction_sets[sheet] = {
        "Core": core_reactions,
        "Accessory": accessory_reactions,
        "Unique": unique_reactions
    }


from openpyxl import load_workbook

with pd.ExcelWriter(file_path, mode="a", engine="openpyxl", if_sheet_exists="overlay") as writer:
    for sheet, sets in reaction_sets.items():
        core_df = pd.DataFrame({"Core Cancer Reactions": list(sets["Core"])})
        accessory_df = pd.DataFrame({"Accessory Cancer Reactions": list(sets["Accessory"])})
        unique_df = pd.DataFrame({"Unique Cancer Reactions": list(sets["Unique"])})
        core_df.to_excel(writer, sheet_name=f"{sheet}_Cancer_Core", index=False)
        accessory_df.to_excel(writer, sheet_name=f"{sheet}_Cancer_Accessory", index=False)
        unique_df.to_excel(writer, sheet_name=f"{sheet}_Cancer_Unique", index=False)


plot_data = {
    "Tissue": [],
    "Core": [],
    "Accessory": [],
    "Unique": []
}
for sheet, sets in reaction_sets.items():
    plot_data["Tissue"].append(sheet)
    plot_data["Core"].append(len(sets["Core"]))
    plot_data["Accessory"].append(len(sets["Accessory"]))
    plot_data["Unique"].append(len(sets["Unique"]))


plot_df = pd.DataFrame(plot_data)
plt.figure(figsize=(12, 8), facecolor='white')
plt.gcf().patch.set_facecolor('white')
bottom = None
for category, color in zip(["Core", "Accessory", "Unique"], ["#1f77b4", "#ff7f0e", "#2ca02c"]):
    if bottom is None:
        plt.bar(plot_df["Tissue"], plot_df[category], label=category, color=color)
        bottom = plot_df[category]
    else:
        plt.bar(plot_df["Tissue"], plot_df[category], bottom=bottom, label=category, color=color)
        bottom += plot_df[category]


plt.xlabel("Tissue", fontsize=16)
plt.ylabel("Number of Reactions", fontsize=15)
plt.title("Core, Accessory, and Unique Cancer Reactions Across Tissues", fontsize=16)
plt.legend(loc="upper left", fontsize=12)
plt.xticks(rotation=45, fontsize=14)
plt.yticks(fontsize=14)
plt.tight_layout()
ax = plt.gca()
ax.spines['bottom'].set_color('black')
ax.spines['left'].set_color('black')
ax.spines['bottom'].set_linewidth(1.5)
ax.spines['left'].set_linewidth(1.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='x', colors='black')
ax.tick_params(axis='y', colors='black')
plt.grid(False)
plt.gca().set_facecolor('white')

plt.savefig('Stack_core_cancer_unique.png', dpi=300, bbox_inches='tight', facecolor='white')# indivual fig can be found in indiviual fig in drive named as same part of panel fig 2 B
plt.show()